### Import Packages

In [80]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

from transformers import BertTokenizerFast
from datasets import load_dataset

import numpy as np
import matplotlib.pyplot as plt
from torchmetrics import Accuracy

from tqdm import tqdm

### 1.1 Load Dataset

In [81]:
from datasets import load_dataset

In [82]:
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

In [83]:
# dataset.save_to_disk(r"../RNN_NLP_Project/datasets")

In [84]:
from datasets import load_from_disk

dataset = load_from_disk(
    r"../RNN_NLP_Project/datasets"
)

In [85]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

### 1.2 Inspect the Dataset 

In [86]:
label_names = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

In [87]:
for i in range(5):
    example = dataset["train"][i]
    print(f"Example {i + 1}")
    print("Text:", example["text"])
    print("Label:", example["label"])
    print("Sentiment:", label_names[example["label"]])
    print("-" * 50)

Example 1
Text: "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"
Label: 2
Sentiment: Positive
--------------------------------------------------
Example 2
Text: "Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"
Label: 1
Sentiment: Neutral
--------------------------------------------------
Example 3
Text: Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.
Label: 1
Sentiment: Neutral
--------------------------------------------------
Example 4
Text: Chase Headley's RBI double in the 8th inning off David Price snapped a Yankees streak of 33 consecutive scoreless innings against Blue Jays
Label: 1
Sentiment: Neutral
--------------------------------------------------
Example 5
Text: @user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017"
Label: 2
Sentiment

### 1.3 Preprocessing

In [88]:
train_dataset = dataset["train"]
valid_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [89]:
train_dataset.shape

(45615, 2)

In [90]:
valid_dataset.shape

(2000, 2)

In [91]:
test_dataset.shape

(12284, 2)

In [92]:
print(train_dataset.column_names)
print(valid_dataset.column_names)
print(test_dataset.column_names)


['text', 'label']
['text', 'label']
['text', 'label']


In [93]:
from transformers import BertTokenizerFast

In [94]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [95]:
tokenizer(["NLP Project Finish Soon"])

{'input_ids': [[101, 17953, 2361, 2622, 3926, 2574, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1]]}

In [96]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128 
    )

### Mapping

In [97]:
train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [98]:
train_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 45615
})

In [99]:
valid_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})

In [100]:
test_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 12284
})

In [101]:
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
valid_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

### DataLoader

In [102]:
BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [103]:
x = next(iter(train_loader))

In [104]:
batch = next(iter(valid_loader))
batch

{'label': tensor([1, 2, 0, 1]),
 'input_ids': tensor([[  101,  2601,  9293,  1017,  2258,  4888,  3058,  4484,  2007,  2047,
           9117,  1024,  9979,  1996,  4768,  1012,   102,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,   

In [105]:
print(batch["input_ids"].shape)

torch.Size([4, 128])


In [106]:
input_ids = x['input_ids'].long()
input_ids

tensor([[  101,  1045,  1005,  1049,  2074,  3201,  2005,  2023,  5353,  1012,
         11660,  5958,  1012,  2125,  5095,  1012,  4463,  2632,  3207,  2319,
          4465,  1012,  2069,  2477,  2000,  2298,  2830,  2000,  1012,   102,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

### 2. Neural Network Model

In [107]:
import torch.nn as nn

In [108]:
vocab_size = tokenizer.vocab_size
vocab_size

30522

In [109]:
embedding = nn.Embedding(30522, 128)
embedding

Embedding(30522, 128)

In [110]:
embd = embedding(input_ids)

In [111]:
embd.shape

torch.Size([4, 128, 128])

In [112]:
# Token IDs
#    │
#    │ [batch, seq_len]
#    ▼
# Embedding
#    │
#    │ [batch, seq_len, 32]
#    ▼
# Bi-RNN
#    │
#    │ hidden
#    ▼
# Final Hidden State
#    │
#    │ [batch, 256]
#    ▼
# Linear
#    │
#    │ [batch, 3]
#    ▼
# ┌──────────┬──────────┬──────────┐
# │ Negative │ Neutral  │ Positive │
# └──────────┴──────────┴──────────┘


class RNNModel(nn.Module):
    def __init__(
        self,
        RNN,
        vocab_size,
        input_size,
        hidden_size,
        num_layers,
        bidirectional,
        num_cls
    ):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, input_size)

        # RNN layer
        self.rnn = RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            batch_first=True
        )

        # Final Linear layer
        self.fc = nn.Linear(
            hidden_size * (2 if bidirectional else 1),
            num_cls
        )

    def forward(self, x):

        # Input Token IDs
        # [batch_size, seq_len]

        # Embedding
        x = self.embedding(x)
        # [batch_size, seq_len, input_size]

        # RNN
        outputs, hidden = self.rnn(x)


        # Average all hidden states
        # [batch, seq_len, hidden_size]
        # x = outputs.mean(dim=1)

        # Final Hidden State
        if self.rnn.bidirectional:
            hidden = torch.cat(
                (hidden[-2], hidden[-1]),
                dim=1
            )
        else:
            hidden = hidden[-1]

        # Linear Classification
        x = self.fc(hidden)

        return x

In [113]:
model = RNNModel(
    nn.RNN,
    vocab_size,
    32, #32 را انتخاب کردیم تا نمایش کلمات نسبتاً مناسب ولی کم‌هزینه باشد.
    128, # 128 را انتخاب کردیم تا RNN ظرفیت کافی برای یادگیری الگوهای متنی و وابستگی‌های بین کلمات داشته باشد، بدون اینکه مدل بیش از حد بزرگ شود.
    1,
    True,
    3 # Negative / Neutral / Positive
)

### 3. Device

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

cuda
Using device: cuda


### 4. Model Training

#### 4.1 Loss and Accuracy

In [115]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [116]:
def train_one_epoch(model, train_loader, loss_fn, optimizer, epoch=None):

    model.train()

    loss_train = AverageMeter()

    acc_train = Accuracy(
        task="multiclass",
        num_classes=3
    ).to(device)

    with tqdm(train_loader, unit="batch") as tepoch:

        for batch in tepoch:

            if epoch is not None:
                tepoch.set_description(f"Epoch {epoch}")

            inputs = batch["input_ids"].to(device)
            targets = batch["label"].to(device)

            outputs = model(inputs)

            loss = loss_fn(outputs, targets)

            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

            loss_train.update(loss.item())

            acc_train.update(outputs, targets)

            tepoch.set_postfix(
                loss=loss_train.avg,
                accuracy=100. * acc_train.compute().item()
            )

    return model, loss_train.avg, acc_train.compute().item()

In [117]:
def validation(model, test_loader, loss_fn):

    model.eval()

    loss_valid = AverageMeter()

    acc_valid = Accuracy(
        task="multiclass",
        num_classes=3
    ).to(device)

    with torch.no_grad():

        for batch in test_loader:

            inputs = batch["input_ids"].to(device)
            targets = batch["label"].to(device)

            outputs = model(inputs)

            loss = loss_fn(outputs, targets)

            loss_valid.update(loss.item())

            acc_valid.update(outputs, targets)

    return loss_valid.avg, acc_valid.compute().item()

In [118]:
model = RNNModel(
    nn.RNN,
    vocab_size=vocab_size,
    input_size=32,
    hidden_size=128,
    num_layers=1,
    bidirectional=True,
    num_cls=3
).to(device)

loss_fn = nn.CrossEntropyLoss()

batch = next(iter(train_loader))

x_batch = batch["input_ids"].to(device)
y_batch = batch["label"].to(device)

outputs = model(x_batch)

print(f"outputs shape: {outputs.shape}")
print(f"targets shape: {y_batch.shape}")

loss = loss_fn(outputs, y_batch)

print(f"loss: {loss.item()}")

outputs shape: torch.Size([4, 3])
targets shape: torch.Size([4])
loss: 1.1630181074142456
